# Data Analysis · Week 15, session 1 of 3
## Series, DataFrame and loading files

**TIA502 · School of Business · Instructor David Escobar-Castillejos**

The spreadsheet you already know how to use exists inside Python, and it is called a
DataFrame. This session introduces the two objects the whole of pandas is built on, and a
two-minute habit that separates a correct analysis from one that merely looks correct.

By the end of this notebook you will be able to:

1. Explain what a `Series` is and why it is the week 12 list with labels on top.
2. Build a `DataFrame` from a dictionary of columns and from a file.
3. Load a CSV in one line with `read_csv`, and know what it did to each column.
4. Inspect a file with `head`, `info`, `shape` and `describe`, always in that order.
5. Spot missing values, duplicated rows and badly captured categories before they ruin a
   result.

### How to use this notebook

Run the cells in order, top to bottom. Several depend on a variable the previous one
defined, so skipping one gives you a `NameError` that has nothing to do with the topic.

Before running a cell marked **Predict**, write your answer down on paper. Getting the
prediction wrong and understanding why teaches more than seeing the right output first
time.

Three cells fail on purpose. They carry a comment saying so, and they catch the error so
the notebook keeps running.

---
## Setup

Two starting cells. The first tells you which pandas version you got, the second puts the
data within reach.

In [ ]:
import pandas as pd

print("pandas", pd.__version__)

### What to expect from your version

Colab updates its libraries when it feels like it, so the number above may be 2.x or 3.x.
For this session it matters for exactly one thing: what the type of a text column is
called.

| | pandas 2.x | pandas 3.0 and later |
|---|---|---|
| A text column reports | `object` | `str` |
| `info()` ends with | `dtypes: float64(1), object(5)` | `dtypes: float64(1), str(5)` |

Same data, same behaviour, different name. `object` was the drawer where pandas kept
anything that was not a number; since version 3.0 text has its own type and no longer
shares a drawer with anyone. If your output says `object` where this notebook says `str`,
nothing went wrong.

In [ ]:
# Plumbing, not the lesson. This cell puts the three course CSVs within
# reach of pandas, and it never needs running again.
#
# It looks for them in the repository first, which is public and reads
# over a URL. If that does not answer, it rebuilds them right here from
# the course's fixed seed, so both routes produce identical files.
# Nothing ever has to be uploaded by hand.
import urllib.request
from pathlib import Path

BASE = ("https://raw.githubusercontent.com/Davidowa/learning-hub/main/"
        "docs/en/courses/python-course/06%20-%20Advanced/data/")
FILES = ["sales.csv", "regions.csv", "employees.csv"]


def _descargar():
    for nombre in FILES:
        with urllib.request.urlopen(BASE + nombre, timeout=15) as r:
            Path(nombre).write_bytes(r.read())


def _reconstruir_datos():
    """Write the three CSV files again from the course's fixed seed.

    They come out byte for byte identical to the ones in the repository,
    so the numbers on the slide still match the ones in the notebook.
    """
    import csv, random
    from datetime import date, timedelta

    rng = random.Random(20260808)
    REGIONS = ["North", "South", "Centre", "West"]
    CHANNELS = ["Retail", "Online", "Wholesale"]
    PRODUCTS = {"Espresso machine": 8990.0, "Coffee grinder": 2450.0,
                "Filter kettle": 1290.0, "Bean subscription": 690.0,
                "Travel mug": 349.0}
    RW = {"North": 1.30, "South": 0.80, "Centre": 1.55, "West": 0.95}
    CW = {"Retail": 1.00, "Online": 1.25, "Wholesale": 2.10}
    MW = [0.72, 0.78, 0.90, 0.95, 1.00, 1.05, 0.98, 0.92, 1.08, 1.15, 1.45, 1.60]

    rows, start = [], date(2025, 1, 6)
    for week in range(52):
        day = start + timedelta(weeks=week)
        for region in REGIONS:
            for _ in range(rng.randint(1, 2)):
                product = rng.choice(list(PRODUCTS))
                channel = rng.choice(CHANNELS)
                base = 9 * RW[region] * CW[channel] * MW[day.month - 1]
                units = max(1, round(rng.gauss(base, base * 0.28)))
                price = PRODUCTS[product] * rng.choice([1.0, 1.0, 1.0, 0.9, 0.85])
                rows.append({"date": day.isoformat(), "region": region,
                             "channel": channel, "product": product,
                             "units": str(units),
                             "unit_price": f"$ {price:,.2f}"})

    # the deliberate dirt: one region typed four ways, blank cells, and
    # rows captured twice
    for i in rng.sample(range(len(rows)), 24):
        rows[i]["region"] = rng.choice(["north", "NORTH", " North", "North "])
    for i in rng.sample(range(len(rows)), 11):
        rows[i]["units"] = ""
    for i in rng.sample(range(len(rows)), 7):
        rows.append(dict(rows[i]))
    rng.shuffle(rows)

    with open("sales.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["date", "region", "channel",
                                          "product", "units", "unit_price"])
        w.writeheader()
        w.writerows(rows)

    with open("regions.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["region", "manager", "country", "monthly_target"])
        w.writerows([["North", "Ana Robles", "Mexico", 480000],
                     ["South", "Luis Ferrer", "Mexico", 300000],
                     ["Centre", "Paula Ines", "Mexico", 560000],
                     ["West", "Marco Duarte", "Mexico", 360000],
                     ["East", "Sofia Lara", "Mexico", 220000]])

    AREAS = {
        "Sales": (["Account executive", "Sales analyst", "Sales manager"], 24000, 62000),
        "Marketing": (["Content specialist", "Campaign analyst", "Brand manager"], 22000, 58000),
        "Finance": (["Accounts clerk", "Financial analyst", "Controller"], 26000, 74000),
        "People": (["Recruiter", "People analyst", "People manager"], 21000, 55000),
        "Operations": (["Warehouse lead", "Logistics analyst", "Operations manager"], 20000, 60000),
    }
    CITIES = ["Mexico City", "Guadalajara", "Monterrey", "Queretaro"]
    emp = []
    for n in range(1, 121):
        area = rng.choice(list(AREAS))
        titles, low, high = AREAS[area]
        idx = rng.choices([0, 1, 2], weights=[5, 3, 1])[0]
        tenure = rng.randint(2, 132)
        salary = round(low + (high - low) * (idx / 2) * rng.uniform(0.82, 1.10)
                       + tenure * 45, -2)
        emp.append({"employee_id": f"E{n:04d}", "area": area,
                    "job_title": titles[idx], "city": rng.choice(CITIES),
                    "tenure_months": tenure, "monthly_salary": int(salary)})
    with open("employees.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(emp[0]))
        w.writeheader()
        w.writerows(emp)


try:
    _descargar()
    print("Data read from the repository.")
except Exception:
    _reconstruir_datos()
    print("The repository did not answer. Data rebuilt in this session.")

print("Ready:", ", ".join(FILES))

---
# Block 1 · The two objects

All of pandas is built on two things, and you already know both of them under another name.
One is the column. The other is the whole sheet.

| In the sheet | In pandas | What you already knew |
|---|---|---|
| One column | `Series` | A list, with an index |
| The whole sheet | `DataFrame` | Several paired lists |
| The row number | The index | The position, starting at 0 |
| The header | `columns` | The keys of a dictionary |

The mapping is exact, and that is why this course can jump straight to pandas without
stopping at an intermediate library.

## A Series is a column

The simplest way to make one is from a list.

In [ ]:
units = pd.Series([15, 8, 22, 5, 11])
print(units)

Two things came back that the list did not have.

On the left there is an **index**, 0 to 4, which pandas created for you. At the bottom
there is a **dtype**, the type shared by every value in the column. A Python list can mix
integers with text; a Series cannot, and nearly all of its speed comes from that.

In [ ]:
print("The index:", list(units.index))
print("The dtype:", units.dtype)
print("The length:", len(units))

The index does not have to be a counter. Give it labels and the Series starts behaving like
a named range in your spreadsheet.

In [ ]:
monthly = pd.Series(
    [42000, 51500, 38900, 60100, 55300, 47800],
    index=["jan", "feb", "mar", "apr", "may", "jun"],
)
print(monthly)

Now a value is reached by its label rather than by counting positions. That is the
difference between writing `monthly["mar"]` and remembering that March was the third one,
or the second if you count from zero.

In [ ]:
print("March:", monthly["mar"])
print("The first three months:")
print(monthly[["jan", "feb", "mar"]])

The summary functions you use in the sheet are methods on the Series. Same arithmetic,
written a different way.

In [ ]:
print("Total:     ", monthly.sum())
print("Average:   ", round(monthly.mean(), 2))
print("Best month:", monthly.idxmax(), "with", monthly.max())
print("Worst month:", monthly.idxmin(), "with", monthly.min())

### The big change: the operation applies to the whole column

An operation on a Series reaches every value at once. There is no `for`, and nothing to
drag down. This is the single biggest change in how you will work from here on.

In [ ]:
with_tax = monthly * 1.16
print(with_tax.round(2))

Comparison works the same way, and gives back a Series of trues and falses. That boolean
Series is what will later filter rows, so it is worth seeing on its own before it appears
inside a bracket.

In [ ]:
good = monthly > 50000
print(good)
print()
print("Months above fifty thousand:", good.sum())
print(monthly[good])

### The edge case: two Series with different indexes

This is where the index stops being decoration. When you add two Series, pandas does not
pair them up by position, it pairs them by label. If a label exists in only one of the two,
the result in that row is `NaN`, which is how pandas writes "no data here".

**Predict before you run.** The first Series has January, February and March. The second
has February, March and April. How many rows does the result have, and what is in each one?

In [ ]:
first = pd.Series([100, 200, 300], index=["jan", "feb", "mar"])
second = pd.Series([10, 20, 30], index=["feb", "mar", "apr"])

print(first + second)

Four rows came back and not three, because pandas kept the union of both sets of labels.
January and April came back as `NaN` because each was missing its partner. February and
March did add up.

This is correct, and it is what you want almost every time, but it surprises people once.
If you expected three numbers and got four with two holes in them, the index is the reason,
not the addition.

## A DataFrame is several Series sharing one index

The usual way to build one is from a dictionary, with one key per column.

In [ ]:
demo = pd.DataFrame({
    "month": ["jan", "feb", "mar", "apr", "may", "jun"],
    "region": ["North", "North", "South", "South", "Centre", "Centre"],
    "amount": [42000, 51500, 38900, 60100, 55300, 47800],
    "units": [15, 18, 12, 21, 19, 16],
})
print(demo)

In [ ]:
print("Shape (rows, columns):", demo.shape)
print("Column names:", list(demo.columns))
print("Index:", list(demo.index))

Each column is a Series, and you pull it out by name. The fact that its type is literally
`Series` is what makes everything from the previous block keep working here.

In [ ]:
print(demo["amount"])
print()
print("Its type:", type(demo["amount"]))

In [ ]:
print(demo.dtypes)

The `dtypes` tell you how pandas understood each column. `month` and `region` hold text,
`amount` and `units` hold integers. This example was built by hand and so it came out
clean. A real file is where the surprises start, and that is what block 3 is about.

### A new column built from the others

Assigning to a name that does not exist yet creates the column. The right-hand side is
computed for every row at once, which is exactly what a formula filled down does.

In [ ]:
demo["price_per_unit"] = (demo["amount"] / demo["units"]).round(2)
print(demo)

And a variant the slide had no room for: the new column can come from a comparison, not
only from a division. Here it marks the months that cleared fifty thousand.

In [ ]:
demo["good_month"] = demo["amount"] > 50000
print(demo[["month", "amount", "good_month"]])
print()
print("How many good ones:", demo["good_month"].sum())

---
# Block 2 · Loading the file

Last week you opened a CSV by hand with the `csv` module: open the file, read the header,
walk the rows and convert each field. `read_csv` does all of that in one line.

It is worth running both versions back to back, because the comparison is the whole argument
for pandas.

## The version by hand, from week 14

In [ ]:
import csv

with open("sales.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

for row in rows:
    row["units"] = int(row["units"] or 0)

print(len(rows), "rows")
print(rows[0])

It works, and every row came back as a dictionary. The problem is not that it is long, it is
that you made a hidden decision: `int(row["units"] or 0)` turned the blank cells into zero.
That changes the average, and nobody is going to notice by reading the code.

## The version with pandas

In [ ]:
sales = pd.read_csv("sales.csv")

print(sales.shape)

One line, and on top of that `read_csv` tried to infer the type of every column. What it
cannot infer is what you wanted a blank cell to mean, which is why it does not fill it with
zero: it marks the cell as missing and leaves the decision to you. That decision is session
15.2.

### One detail that pays off all term

`read_csv` takes a URL anywhere it takes a path. With the repository public, loading the
course data from any machine looks like this, with nothing downloaded by hand:

```python
BASE = ("https://raw.githubusercontent.com/Davidowa/learning-hub/main/"
        "docs/en/courses/python-course/06%20-%20Advanced/data/")

sales = pd.read_csv(BASE + "sales.csv")
```

The setup cell in this notebook tries exactly that before rebuilding the files on its own.

## The first rows

In [ ]:
print(sales.head())

`head` shows the first five rows. Pass it a number to change how many.

In [ ]:
print(sales.head(3))

`tail` shows the last ones, and it is the one to run even when it feels redundant. That is
where the totals row somebody pasted at the bottom of the sheet before exporting it hides,
and if you do not see it, it ends up counted as one more sale.

In [ ]:
print(sales.tail(3))

`shape` answers how much data you actually have. It returns a tuple, so it can be unpacked
into two names.

In [ ]:
rows_n, columns_n = sales.shape
print(f"{rows_n} rows and {columns_n} columns")

---
# Block 3 · Look before touching

The temptation is to jump straight to the answer. Hold off for two minutes. Almost every
wrong result in the final project traces back to a column that was not the type you
assumed, or to rows that were not there.

The order is always the same: `head`, `info`, `shape`, `describe`.

## The types it inferred

In [ ]:
print(sales.dtypes)

`info` is the most useful command in this session. It reports, per column, how many values
are not empty and what type pandas inferred. Watch the `Non-Null Count` column: one that
does not reach the total number of rows has holes in it.

In [ ]:
sales.info()

### What those types are telling you

Three things in that output deserve to be named.

**`units` came in as `float64` and not as a whole number.** pandas read the digits fine, but
eleven cells are blank, and a blank has to be represented somehow. That marker is `NaN`,
which only exists in a decimal column, so the whole column became decimal. This is why the
counts print as `15.0` instead of `15`.

**`unit_price` came in as text**, because `"$ 2,082.50"` is not a number to Python. The
currency symbol and the thousands comma are formatting, and formatting is not part of the
value.

**`date` came in as text**, because a CSV has no date type. Until it is converted, sorting
by that column works by luck: it only comes out right because the format puts the year
first.

None of the three is a pandas failure. It is the CSV, which stores no types, exactly as you
saw last week. Session 15.2 fixes all three.

## The numeric summary

In [ ]:
print(sales.describe())

`describe` gives count, mean, standard deviation, minimum, maximum and the quartiles for
every numeric column. So far only `units` qualifies, and its count of 313 against 324 rows
is the missing data showing up again.

That is why `describe` says so little on a freshly loaded file: it is not that there are no
numbers, it is that they are stored as text. With `include="all"` pandas summarises the text
columns too, using different statistics.

In [ ]:
print(sales.describe(include="all"))

Two rows of that table are worth a look. `unique` counts how many distinct values the column
holds, and `top` names the one that repeats most. `region` reports eight distinct values,
and the company has four regions.

## Finding the dirt

For a text column, what helps is counting how many times each value appears.

In [ ]:
print(sales["region"].value_counts())

Four regions were captured and the file thinks there are eight, because the same name was
typed with different capitalisation and with stray spaces. `" North"`, `"North "`, `"north"`
and `"NORTH"` are, to Python, four strings with nothing to do with each other.

Group by region today and the north splits into five pieces, none of which carries the real
total. Session 15.2 fixes it, and today's point is that you catch it before it happens.

In [ ]:
print("Distinct values in region:", sales["region"].nunique())
print("Distinct values in channel:", sales["channel"].nunique())
print("Distinct values in product:", sales["product"].nunique())

`isna` marks every missing cell as true, and `sum` counts them per column.

In [ ]:
print(sales.isna().sum())
print()
print("Missing across the whole table:", sales.isna().sum().sum())

`duplicated` marks a row as true when an identical row appeared earlier. The seven here are
the trace of a copy and paste.

In [ ]:
print("Duplicated rows:", sales.duplicated().sum())
print()
print(sales[sales.duplicated(keep=False)].sort_values("date").head(6))

Note the `keep=False` on that last line: with that argument pandas marks every copy, not
only the repeats, so you can see the full pairs and confirm they really are identical.

And mind the arithmetic. 324 rows with seven duplicates are 317 distinct facts. Which of the
two numbers belongs in your report depends on what you are counting, and deciding that is
your job, not pandas'.

---
## Three cells that fail on purpose

A wrong type does not always blow up. Sometimes it hands you a number, and that is the
dangerous case.

### The one that does not fail, which is why it is the worst

In [ ]:
# FAILS ON PURPOSE. This cell raises nothing at all, and that is exactly the problem.
# unit_price is text, so sum() concatenates instead of adding.
total = sales["unit_price"].sum()

print("Type of the result:", type(total))
print("First 70 characters:", str(total)[:70])

You asked for a sales total and received all 324 prices glued one after another into a
single string. No error, no warning. If this sits inside a longer report, it gets published.

This is the entire argument for running `info()` before analysing. Two minutes of inspection
against a wrong number in a presentation.

### The one that does fail, and warns you in time

In [ ]:
# FAILS ON PURPOSE. Averaging text does raise, unlike adding it.
try:
    sales["unit_price"].mean()
except TypeError as e:
    print("TypeError:", e)

`mean` has no way to invent an average of strings, so it stops. Same problem as the cell
before, with better luck: here the error shows up while you are writing it, not three weeks
later in a meeting.

### The conversion that looks obvious and is not

In [ ]:
# FAILS ON PURPOSE. units carries eleven NaN, and NaN does not fit in an integer.
try:
    sales["units"].astype(int)
except Exception as e:
    print(type(e).__name__ + ":", e)

If seeing `15.0` where you expected `15` bothered you, this is the natural attempt to fix
it, and it does not work. You cannot convert to integer while there are missing values,
because `NaN` is not a representable whole number.

The right order is the opposite of how it feels: first you decide what the eleven holes
mean, then you convert. Filling with zero, dropping those rows, or leaving them missing are
three different decisions with three different averages, and none of them is the default.

## The second table

In [ ]:
regions = pd.read_csv("regions.csv")
print(regions)

This is the course's lookup table, the equivalent of the `VLOOKUP` in your sheet: one row per
region, with the details that have no business repeating in every sales record.

It carries five regions and the sales file only covers four. The difference is deliberate,
and session 15.3 shows what a join does with the region that has no sales.

---
## Predict before you run

Write your answer down before executing each cell.

### Question 1

Why did the `units` column come back as `float64` and not `int64`?

- **A.** Because the unit counts carry decimals in the file.
- **B.** Because eleven blank cells need `NaN`, which only exists in float.
- **C.** Because `read_csv` always uses float to be safe.
- **D.** Because the column has more than three hundred rows.

In [ ]:
print("dtype of units:      ", sales["units"].dtype)
print("Missing in units:    ", sales["units"].isna().sum())
print("Any value with a decimal part?")
print((sales["units"].dropna() % 1 != 0).sum(), "out of", sales["units"].notna().sum())

Zero values with a decimal part, and eleven missing. The answer is **B**: the numbers were
whole from the start, and what forced the decimal type was the marker for blank.

### Question 2

`sales.shape` said 324 rows. How many distinct sales does the file describe, and how many
have their unit count on record?

In [ ]:
print("Rows:                     ", len(sales))
print("Not counting duplicates:  ", len(sales.drop_duplicates()))
print("With units recorded:      ", sales["units"].notna().sum())
print("Distinct and with units:  ", len(sales.drop_duplicates().dropna(subset=["units"])))

Four different numbers, all correct, all answers to different questions. Which one to use
depends on what you claim in the report, and that is why `shape` is not "the number of data"
but the first of several.

### Question 3

What does the next cell print? Think about what type `sales["units"]` has and what type
`sales[["units"]]` has.

In [ ]:
print(type(sales["units"]))
print(type(sales[["units"]]))
print()
print(sales[["date", "region", "units"]].head(3))

One bracket gives back a `Series`, the column on its own. Two brackets give back a
`DataFrame`, because what you passed was a list of names and a list can carry more than one.
It is the most common confusion of the next two sessions, and you have now seen it with both
types printed out.

---
## Four errors when loading a file

**Analysing before inspecting.** The result comes out, it looks reasonable, and it is wrong.
`head`, `info` and `describe` cost two minutes and are the only defence against the bug
nobody finds because nobody is looking for it.

**Trusting the inferred type.** pandas guesses right nearly every time. That *nearly* is
where the price column that turned out to be text lives.

**Not checking the categories.** `value_counts` on a text column exposes inconsistent capture
before it splits your groups into five pieces.

**Taking `shape` as the number of facts.** 324 rows with seven duplicates are 317 distinct
facts, and the total changes with which one you count.

---
# Exercises

Solve them in new cells below each brief. The solutions sit at the very bottom of the
notebook, so you cannot catch them out of the corner of your eye while you work.

They run from lighter to heavier. The first four repeat what you just saw with different
data; the next three ask you to combine two ideas; the last one is about your own file.

## Warming up

### Exercise 1 · A Series with labels

Build a `Series` with the sales for the first six months of the year, using the month labels
as the index. Print the total, the average rounded to two decimals, and the name of the
weakest month. Then add 8 % to every month in a single operation.

### Exercise 2 · A DataFrame from scratch

Build a `DataFrame` with five products from a coffee shop: name, unit price and pieces sold
during the week. Add an `income` column that multiplies price by pieces, print the table
sorted from highest to lowest income, and say how much was sold in total.

Hint: `df.sort_values("income", ascending=False)`.

### Exercise 3 · The full diagnosis

Write a function `diagnose(df)` that takes a DataFrame and prints, in this order: how many
rows and columns it has, what type each column holds, how many values are missing per
column, and how many duplicated rows there are. Test it with `sales` and with `regions`.

### Exercise 4 · The text columns

Walk the columns of `sales` and, for each one that came back as text, print the name and how
many distinct values it holds. Then say, in a comment, which of those columns should be
converted to another type and which are fine as text.

## Worth thinking about

### Exercise 5 · What the missing value costs

The eleven rows with no unit count have three possible fates: fill them with zero, drop
them, or leave them alone. Work out the average of `units` under all three decisions and
print them in the same output.

Then answer in a comment which one you would use if the report says "average units per
sale", and why the other two would be wrong there.

Hints: `.fillna(0)`, `.dropna()`, and `.mean()`, which already ignores missing values on its
own.

### Exercise 6 · The size of the mess

Without cleaning anything yet, measure how much damage analysing the file as it stands would
do. Count how many rows carry a dirty version of `"North"`, meaning any `region` value that
is not exactly one of the four correct names.

Then print, side by side, how many rows the file thinks are northern and how many really
are.

Hint: `~sales["region"].isin([...])` inverts a membership test.

### Exercise 7 · The highest paid employee

Load `employees.csv`, which has 120 rows and comes out clean. Print how many areas there
are, how many people each one holds, and the highest monthly salary in the table along with
the identifier of whoever earns it.

You do not need to group yet, that is session 15.3. `value_counts`, `max` and `idxmax` are
enough, and that is exactly the point of the exercise.

## With your own data

### Exercise 8 · Your own file

Load your project CSV with pandas and write a half-page diagnosis covering how many rows it
has, what type each column was inferred as, how many values are missing and how many
duplicates there are.

Do not clean anything yet. Today is only looking and writing down. For every column that
came back as text, say whether that is right or whether something needs converting.

---
## Three ideas to take away

**A Series is a column with an index.** It is the week 12 list with labels, and everything
you learned there still holds here. The index is not decoration: it is what lines the data
up when you combine two objects.

**`read_csv` cannot guess what it cannot guess.** It infers types well nearly always, and it
has no way of knowing what you wanted a blank cell to mean. That decision is yours and it
changes the result.

**Inspect before analysing.** `head`, `info`, `shape` and `describe`. Two minutes that
prevent a wrong result that looks reasonable, which is the worst kind of wrong result.

Next session is selecting, filtering and cleaning. That is where everything diagnosed today
gets fixed.

---
# Solutions

Compare them with yours after you have tried. If your version reaches the same result by a
different route, that is fine: there is no single correct way here.

### Exercise 1

```python
monthly_sales = pd.Series(
    [42000, 51500, 38900, 60100, 55300, 47800],
    index=["jan", "feb", "mar", "apr", "may", "jun"],
)

print("Total:", monthly_sales.sum())
print("Average:", round(monthly_sales.mean(), 2))
print("Weakest month:", monthly_sales.idxmin(), "with", monthly_sales.min())

with_raise = monthly_sales * 1.08
print(with_raise.round(2))
```

The raise applies to all six months in a single line. No loop is needed, and writing one
here is the most common sign that somebody is still thinking in lists.

### Exercise 2

```python
shop = pd.DataFrame({
    "product": ["Americano", "Cappuccino", "Latte", "Sweet bread", "Croissant"],
    "price": [38.0, 52.0, 55.0, 24.0, 46.0],
    "pieces": [310, 185, 142, 260, 98],
})

shop["income"] = shop["price"] * shop["pieces"]
print(shop.sort_values("income", ascending=False))
print("\nTotal income:", shop["income"].sum())
```

`sort_values` gives back a new table and leaves the original untouched. If you wanted the
change to stick, you have to reassign: `shop = shop.sort_values(...)`. Nearly every pandas
method behaves this way, and that is one of the reasons session 15.2 opens by talking about
copies.

### Exercise 3

```python
def diagnose(df):
    rows_n, columns_n = df.shape
    print(f"{rows_n} rows and {columns_n} columns")

    print("\nTypes per column:")
    print(df.dtypes)

    print("\nMissing per column:")
    print(df.isna().sum())

    print("\nDuplicated rows:", df.duplicated().sum())


diagnose(sales)
print("\n" + "=" * 40 + "\n")
diagnose(regions)
```

`regions` comes out clean: five rows, nothing missing, nothing duplicated. That contrast is
useful, because it shows what a healthy file looks like and gives you something to compare
against.

### Exercise 4

```python
for name in sales.columns:
    if sales[name].dtype == "object" or sales[name].dtype == "str":
        print(f"{name:12} {sales[name].nunique():4} distinct values")

# date        should be converted to a date, so it can be sorted and grouped by month
# region      should be normalised to four values, not converted to another type
# channel     fine as text, three values and all of them consistent
# product     fine as text, five values and all of them consistent
# unit_price  should be converted to a number, dropping the sign and the comma
```

Comparing against both `"object"` and `"str"` covers the two pandas versions. Compare against
only one and the exercise works in your Colab session and fails in your classmate's.

### Exercise 5

```python
print("Filling with zero:", round(sales["units"].fillna(0).mean(), 2))
print("Dropping them:    ", round(sales["units"].dropna().mean(), 2))
print("Leaving them be:  ", round(sales["units"].mean(), 2))

print("\nRows that enter each count:")
print("Filling with zero:", sales["units"].fillna(0).count())
print("Dropping them:    ", sales["units"].dropna().count())

# For "average units per sale" the answer is 16.12, so drop them or leave them.
# Filling with zero invents eleven sales of zero units that never happened, and
# drags the average down for a reason that is not in the data.
```

You get 15.57 with zeros and 16.12 in the other two cases. The gap looks small until you
multiply it by the annual volume.

Notice something surprising: dropping them and leaving them alone give the **same average**,
because `mean` already ignores missing values on its own. What does change between those two
is the count, 313 against 313 here, but the moment you sum or divide by `len(df)` they start
to diverge. That is why the decision has to be made out loud instead of trusting whatever
the method does.

### Exercise 6

```python
CORRECT = ["North", "South", "Centre", "West"]

dirty = ~sales["region"].isin(CORRECT)
print("Rows with a badly captured region:", dirty.sum())
print(sales.loc[dirty, "region"].value_counts())

print("\nWhat the file thinks:", (sales["region"] == "North").sum())
print("What is actually true:",
      sales["region"].str.strip().str.title().eq("North").sum())
```

Twenty-four rows carry a dirty version. The real north has 99 sales and the file reports 75,
so a report written today takes a quarter of the north's volume away and hands it to four
phantom regions.

`.str.strip().str.title()` is a preview of session 15.2, and it is used here only to measure.
Fixing the file is next class.

### Exercise 7

```python
employees = pd.read_csv("employees.csv")

print(employees.shape)
print("\nAreas:", employees["area"].nunique())
print(employees["area"].value_counts())

highest = employees["monthly_salary"].idxmax()
print("\nHighest salary:", employees["monthly_salary"].max())
print("Earned by:", employees.loc[highest, "employee_id"],
      "in", employees.loc[highest, "area"])
```

Five areas, 120 people, and the highest salary is 82,700, earned by `E0003`.

What makes this exercise worth doing is `idxmax`. It gives back the **row label** where the
maximum sits, not the maximum itself, and with that label `.loc` brings you the whole row.
It is the "who has the highest value" pattern, and you will use it in every report this
term.

### Exercise 8

There is no published solution, because the file is different for everyone. The diagnosis is
graded on four things: that the four requested numbers are there, that you name the columns
that came back as text, that you say for each one whether that is right, and that you have
not cleaned anything yet.